In [5]:
import pandas as pd
import numpy as np
import os
import datetime
import warnings
import tkinter as tk
from tkinter import filedialog
import re

# Silenciar advertencias
warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURACIÓN DE RUTAS Y REGLAS (SELECCIÓN MANUAL)
# =============================================================================

def seleccionar_archivo(titulo):
    """Abre ventana para seleccionar archivo Excel"""
    print(f"Esperando selección: {titulo}...")
    ruta = filedialog.askopenfilename(
        title=titulo,
        filetypes=[("Archivos Excel", "*.xlsx *.xls *.xlsm")]
    )
    if not ruta:
        print(f"⚠️ No se seleccionó ningún archivo para {titulo}.")
        return "" # Retorna vacío si cancelas
    print(f"✅ Seleccionado: {ruta}")
    return ruta

def seleccionar_carpeta(titulo):
    """Abre ventana para seleccionar carpeta"""
    print(f"Esperando selección: {titulo}...")
    ruta = filedialog.askdirectory(title=titulo)
    if not ruta:
        print(f"⚠️ No se seleccionó carpeta.")
        return ""
    print(f"✅ Carpeta: {ruta}")
    return ruta

# Inicializar Tkinter y ocultar la ventana principal
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True) # Forzar que las ventanas salgan al frente

print("\n=== SELECCIÓN DE ARCHIVOS ===")

# 1. Seleccionar Archivos de Entrada
path_turnos   = seleccionar_archivo("Selecciona el archivo de TURNOS")
path_mp       = seleccionar_archivo("Selecciona el archivo de MERCADO PAGO")
path_nave     = seleccionar_archivo("Selecciona el archivo de FISERV")
path_ventas   = seleccionar_archivo("Selecciona el archivo de VENTAS (Sistema)")
path_adicion  = seleccionar_archivo("Selecciona el archivo de ADICIÓN")
path_descuento = seleccionar_archivo("Selecciona el archivo de DESCUENTOS")
path_auditoria = seleccionar_archivo("Selecciona el archivo de AUDITORÍA") 


# 2. Seleccionar el Macro
path_macro    = seleccionar_archivo("Selecciona el archivo MACRO (.xlsm)")

# 3. Seleccionar Carpeta de Salida y definir nombre del archivo final
folder_salida = seleccionar_carpeta("Selecciona la CARPETA donde guardar el resultado")
if folder_salida:
    path_output_final = os.path.join(folder_salida, "Resultado_Conciliacion.xlsx")
else:
    path_output_final = "Resultado_Conciliacion.xlsx" # Fallback por si cancelas

print("=============================\n")

# Reglas originales
REGLAS_CONCILIACION = [
    ("R0: 1min / $1", 1, 1),        
    ("R1: 5min / $5", 5, 5),        
    ("R2: 30min / $5", 30, 5),
    ("R3: Mismo Día / $5", 1440, 5) 
]

# Medios de Fiserv
MEDIOS_FISERV = [
    'VISADEBITO', 'MASTER', 'QRMODO', 'VISA', 'AMEX', 
    'CABAL', 'MASTERDEBITO', 'MAESTRO', 'NARANJA'
]

# Palabras clave para excluir
KEYWORDS_EXCLUIR = ['interno', 'ingreso extra', 'caja chica', 'socio']

# Mapeo de búsqueda
TARGET_QR = "QRMERC.PAGO"
TARGET_EFECTIVO = "EFECTIVO"
TARGET_CTACTE = "CUENTACORRIENTE"


# =============================================================================
# 2. FUNCIONES AUXILIARES (FECHAS, TURNOS Y TEXTO)
# =============================================================================

# =============================================================================
# 2. FUNCIONES AUXILIARES (FECHAS, TURNOS Y TEXTO)
# =============================================================================

def parsear_fecha_mp_iso(series):
    s = series.astype(str).str.strip()
    s = s.str.slice(0, 19).str.replace("T", " ", regex=False)
    fechas = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    if fechas.isna().sum() > (len(fechas) * 0.5): 
        fechas = pd.to_datetime(s, dayfirst=True, errors='coerce')
    return fechas

def combinar_fecha_hora_sistema(row):
    if pd.isna(row['FECHA']): 
        return pd.NaT
    
    hora_final = datetime.time(0, 0, 0) # Por defecto
    hora_input = row['HORA']
    
    # --- 1. PROCESAR EL TEXTO DE LA HORA (Igual que antes) ---
    if pd.notna(hora_input):
        try:
            if isinstance(hora_input, datetime.datetime): 
                hora_final = hora_input.time()
            elif isinstance(hora_input, datetime.time): 
                hora_final = hora_input
            elif isinstance(hora_input, (float, int)): 
                # Decimal de Excel (ej: 0.5 es 12:00hs)
                total_seconds = int(float(hora_input) * 24 * 3600)
                hora_final = (datetime.datetime.min + datetime.timedelta(seconds=total_seconds)).time()
            elif isinstance(hora_input, str):
                hora_str = hora_input.strip()
                try: 
                    hora_final = datetime.datetime.strptime(hora_str, "%H:%M:%S").time()
                except: 
                    try: 
                        hora_final = datetime.datetime.strptime(hora_str, "%H:%M").time()
                    except: 
                        pass
        except: 
            pass

    # --- 2. APLICAR LÓGICA EXCEL (LAS 6:00 AM) ---
    # Lógica: Si la hora es menor a las 06:00 AM, asumimos que es del día siguiente (Madrugada)
    # CONFIG!$Y$5 = 06:00 AM
    
    fecha_base = row['FECHA'].date()
    hora_corte = datetime.time(6, 0, 0) # 06:00:00 AM
    
    if hora_final < hora_corte:
        # Es de madrugada (ej: 02:00 AM), sumamos 1 día a la fecha del turno
        fecha_base = fecha_base + datetime.timedelta(days=1)
        # print(f"  [DEBUG] Rollover aplicado: {row['FECHA'].date()} -> {fecha_base} por hora {hora_final}")

    # 3. Combinar fecha corregida + hora
    dt_combinado = datetime.datetime.combine(fecha_base, hora_final)
    
    return dt_combinado

def calcular_mascara_exclusion(series_clasificacion):
    """Detecta si la clasificación contiene palabras clave como 'interno'"""
    s = series_clasificacion.fillna('').astype(str).str.lower().str.strip()
    return s.isin(KEYWORDS_EXCLUIR)

def identificar_plataforma_real(medio_cobro):
    """
    Identifica si un medio de cobro pertenece a fiserv.
    Retorna 'fiserv' si es un medio de fiserv, sino retorna cadena vacía.
    """
    if pd.isna(medio_cobro):
        return ''
    
    medio_normalizado = str(medio_cobro).upper().strip().replace(' ', '')
    
    # Verificar si contiene alguna palabra clave de fiserv
    for medio_nave in MEDIOS_FISERV:
        if medio_nave in medio_normalizado:
            return 'Fiserv'
    
    return ''

def asignar_turno_desde_excel(df_reporte, col_fecha_dt, df_turnos_maestro):
    """
    MODIFICADO: Ahora asigna TURNO y FECHA_TURNO (Fecha de Apertura del Turno).
    Si es un turno de madrugada (ej 03/01 02:00am que pertenece al turno del 02/01),
    FECHA_TURNO será 02/01.
    """
    if df_turnos_maestro.empty:
        df_reporte['TURNO'] = "Sin Data Turnos"
        df_reporte['FECHA_TURNO'] = pd.NaT
        return df_reporte
    
    # 1. Convertir fecha si no lo es
    if not pd.api.types.is_datetime64_any_dtype(df_reporte[col_fecha_dt]):
         df_reporte[col_fecha_dt] = pd.to_datetime(df_reporte[col_fecha_dt], dayfirst=True, errors='coerce')
    
    # 2. Limpieza de zona horaria
    try:
        df_reporte[col_fecha_dt] = df_reporte[col_fecha_dt].dt.tz_localize(None)
    except: 
        pass
    
    # 3. Función de búsqueda MEJORADA (retorna turno Y fecha)
    def find_data_turno(transaction_time):
        if pd.isna(transaction_time): 
            return "Fecha Nula", pd.NaT
        
        # Busca si la hora cae en algún turno
        mask = (transaction_time >= df_turnos_maestro['Apertura_DT']) & \
               (transaction_time <= df_turnos_maestro['Cierre_DT'])
        matching = df_turnos_maestro.loc[mask]
        
        if not matching.empty: 
            # Retorna: (Nombre del Turno, Fecha de Apertura de ese Turno)
            return matching.iloc[0]['TURNO'], matching.iloc[0]['Fecha Apertura']
        
        # Si no encuentra turno, devuelve fecha real
        return "Fuera de turno", pd.to_datetime(transaction_time.date())

    # 4. Aplicamos la búsqueda fila por fila
    resultados = df_reporte[col_fecha_dt].apply(find_data_turno)
    
    # 5. Guardamos los dos datos en columnas separadas
    df_reporte['TURNO'] = [res[0] for res in resultados]
    df_reporte['FECHA_TURNO'] = [res[1] for res in resultados]
    
    # 6. Aseguramos que sea fecha y no texto
    df_reporte['FECHA_TURNO'] = pd.to_datetime(df_reporte['FECHA_TURNO'], errors='coerce')

    return df_reporte

def procesar_archivo_adicion(ruta):
    print(">>> Procesando Archivo de Adición (Limpieza TURNO)...")
    if not os.path.exists(ruta):
        print(f"ADVERTENCIA: No se encontró el archivo de adición en {ruta}")
        return pd.DataFrame()

    try:
        df = pd.read_excel(ruta, sheet_name="Hoja1")
        
        # 1. LIMPIAR NOMBRES DE COLUMNAS (Para encontrar "TURNO" seguro)
        # Esto convierte " TURNO " o "Turno" en "TURNO"
        df.columns = df.columns.astype(str).str.strip().str.upper()

        # 2. LIMPIEZA AGRESIVA DE LA COLUMNA TURNO
        if 'TURNO' in df.columns:
            print("   -> Limpiando columna TURNO...")
            
            def limpiar_turno_forzado(valor):
                # Convertimos a texto mayúscula para analizar
                texto = str(valor).upper()
                
                # SI CONTIENE LA PALABRA, ESCRIBE EL VALOR LIMPIO
                if "TARDE" in texto:
                    return "TARDE"
                elif "MAÑANA" in texto or "MANANA" in texto:
                    return "MAÑANA"
                else:
                    # Si no dice ni Mañana ni Tarde, devolvemos limpio de espacios
                    return texto.strip()

            df['TURNO'] = df['TURNO'].apply(limpiar_turno_forzado)
        else:
            print("   [ALERTA] No se encontró la columna TURNO en el archivo de Adición.")

        # ---------------------------------------------------------
        # Resto del procesamiento normal
        # ---------------------------------------------------------
        
        # Formatos de Fecha e Importe
        if 'FECHA' in df.columns:
            df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')
        
        if 'IMPORTE' in df.columns:
            df['IMPORTE'] = pd.to_numeric(df['IMPORTE'], errors='coerce')

        # Lógica del Total Recuento (Text.AfterDelimiter)
        def obtener_texto_despues_delimitador(texto):
            if pd.isna(texto): return ""
            texto_str = str(texto)
            if ":" in texto_str:
                return texto_str.split(':', 1)[1].strip()
            return ""

        if 'DETALLE' in df.columns:
            df['Total Recuento'] = df['DETALLE'].apply(obtener_texto_despues_delimitador)
            df['Total Recuento'] = df['Total Recuento'].replace("", "0")
            df['Total Recuento'] = df['Total Recuento'].astype(str).str.replace(',', '.')
            df['Total Recuento'] = pd.to_numeric(df['Total Recuento'], errors='coerce').fillna(0)
        else:
            df['Total Recuento'] = 0

        return df

    except Exception as e:
        print(f"ERROR al procesar archivo de adición: {e}")
        return pd.DataFrame()
    
def procesar_cruce_descuentos(df_desc, df_sys):
    print(">>> Cruzando Descuentos con Sistema para obtener Turnos...")
    
    if df_desc.empty or df_sys.empty:
        if not df_desc.empty:
            df_desc['TURNO'] = "No se encontró"
        return df_desc

    # 1. Crear LLAVE en DESCUENTOS (Quitamos todos los espacios y ponemos mayúsculas)
    # Ejemplo: " FCB 001 " -> "FCB001"
    df_desc['Key_Link'] = df_desc['COMPROBANT'].astype(str).str.replace(' ', '').str.strip().str.upper()

    # 2. Crear LLAVE en SISTEMA (Igual proceso)
    # Solo nos interesan filas que tengan turno asignado
    df_sys_clean = df_sys[df_sys['TURNO'].notna()].copy()
    df_sys_clean['Key_Link'] = df_sys_clean['COMPROBANT'].astype(str).str.replace(' ', '').str.strip().str.upper()

    # 3. Crear Diccionario de Mapeo: {ComprobanteLimpio : TURNO}
    # drop_duplicates para evitar errores si el sistema tuviera items repetidos (toma el primero)
    mapa_turnos = df_sys_clean.drop_duplicates(subset=['Key_Link']).set_index('Key_Link')['TURNO'].to_dict()

    # 4. Cruzar
    df_desc['TURNO'] = df_desc['Key_Link'].map(mapa_turnos)

    # 5. Rellenar los que no cruzaron
    df_desc['TURNO'] = df_desc['TURNO'].fillna("No se encontró")
    
    # 6. Limpieza final (borramos la columna auxiliar)
    df_desc = df_desc.drop(columns=['Key_Link'], errors='ignore')
    
    print(f"    -> Turnos asignados en Descuentos. (Encontrados: {len(df_desc[df_desc['TURNO'] != 'No se encontró'])})")
    
    return df_desc



# =============================================================================
# 3. CARGA Y PREPARACIÓN DE DATOS
# =============================================================================
def procesar_archivo_auditoria(ruta):
    print(">>> Procesando Archivo de Auditoría...")
    if not ruta or not os.path.exists(ruta):
        print("   [AVISO] No hay archivo de auditoría seleccionado.")
        return pd.DataFrame()

    try:
        # Leer sin encabezados
        df = pd.read_excel(ruta, header=None)
        
        # Quedarse solo con la primera columna
        df = df.iloc[:, [0]].copy()
        df.columns = ['Raw']

        # 1. Extraer Fecha y Turno (Regex)
        patron_encabezado = r'Fecha (\d{2}/\d{2}/\d{4}) Turno (\d+)'
        extraction = df['Raw'].astype(str).str.extract(patron_encabezado)
        df['Fecha'] = extraction[0]
        df['Turno_Raw'] = extraction[1]

        # 2. Rellenar hacia abajo
        df['Fecha'] = df['Fecha'].ffill()
        df['Turno_Raw'] = df['Turno_Raw'].ffill()

        # 3. Limpiar encabezados viejos
        df_clean = df[~df['Raw'].astype(str).str.startswith('Fecha', na=False)].copy()
        df_clean = df_clean.dropna(subset=['Raw'])

        # 4. Formatos
        df_clean['Fecha'] = pd.to_datetime(df_clean['Fecha'], dayfirst=True).dt.date
        mapeo_turnos = {'1': 'MAÑANA', '2': 'TARDE'}
        df_clean['TURNO'] = df_clean['Turno_Raw'].map(mapeo_turnos).fillna(df_clean['Turno_Raw'])

        # 5. Separar Hora y Detalle
        patron_venta = r'(\d{2}:\d{2})\s+(.*)'
        ventas_ext = df_clean['Raw'].astype(str).str.extract(patron_venta)
        df_clean['Hora'] = ventas_ext[0]
        df_clean['Detalle'] = ventas_ext[1]

        # 6. Final
        cols_final = ['Fecha', 'TURNO', 'Hora', 'Detalle', 'Raw']
        df_final = df_clean[cols_final].copy()
        
        print(f"   -> Auditoría procesada: {len(df_final)} registros.")
        return df_final

    except Exception as e:
        print(f"   [ERROR] Procesando auditoría: {e}")
        return pd.DataFrame()

def cargar_datos_normalizados():
    print(">>> 1. Cargando y Normalizando Tablas...")
    
    # =========================================================================
    # A. TURNOS (CORREGIDO Y BLINDADO CON FECHA_TURNO)
    # =========================================================================
    try:
        print("   -> Cargando archivo de Turnos...")
        df_t = pd.read_excel(path_turnos, sheet_name=0)
        
        # 1. Aseguramos que Fecha Apertura y Cierre sean Datetime puro
        df_t['Fecha Apertura'] = pd.to_datetime(df_t['Fecha Apertura'], dayfirst=True, errors='coerce')
        df_t['Fecha Cierre'] = pd.to_datetime(df_t['Fecha Cierre'], dayfirst=True, errors='coerce')
        
        # *** IMPORTANTE: Normalizar a solo fecha (sin hora) ***
        try:
            df_t['Fecha Apertura'] = df_t['Fecha Apertura'].dt.normalize()
            df_t['Fecha Cierre'] = df_t['Fecha Cierre'].dt.normalize()
        except:
            pass

        # 2. Función auxiliar para limpiar la hora
        def limpiar_y_convertir_hora(val):
            if pd.isna(val): return "00:00:00"
            # Si es objeto time
            if isinstance(val, datetime.time):
                return val.strftime("%H:%M:%S")
            # Si es texto o número
            s = str(val).strip()
            if len(s) == 5 and ":" in s: return s + ":00"
            return s

        # Aplicamos la limpieza
        df_t['Hora_Ap_Str'] = df_t['Hs Ap. Caja'].apply(limpiar_y_convertir_hora)
        df_t['Hora_Ci_Str'] = df_t['Hs Cierre Caja'].apply(limpiar_y_convertir_hora)

        # 3. Combinar
        df_t['Apertura_DT'] = pd.to_datetime(
            df_t['Fecha Apertura'].dt.strftime('%Y-%m-%d') + ' ' + df_t['Hora_Ap_Str'],
            errors='coerce'
        )
        
        df_t['Cierre_DT'] = pd.to_datetime(
            df_t['Fecha Cierre'].dt.strftime('%Y-%m-%d') + ' ' + df_t['Hora_Ci_Str'],
            errors='coerce'
        )

        df_t['TURNO'] = df_t['TURNO'].astype(str).str.upper().str.strip()
        
        # DIAGNÓSTICO EN CONSOLA
        print("   [INFO] Turnos cargados. Muestra:")
        print(df_t[['TURNO', 'Fecha Apertura', 'Apertura_DT', 'Cierre_DT']].head(3))
        
        if df_t['Apertura_DT'].isna().all():
            print("   [ALERTA] Falló la conversión de fechas en Turnos. Revisa el Excel.")

    except Exception as e:
        print(f"   [ERROR] Cargando turnos: {e}")
        df_t = pd.DataFrame()

    # =========================================================================
    # B. MERCADO PAGO (VERSIÓN BLINDADA CON XLWINGS)
    # =========================================================================
    df_mp = pd.DataFrame()
    try:
        print("   -> Cargando archivo de Mercado Pago...")
        leido = False

        # --- INTENTO 1: LECTURA RÁPIDA (PANDAS) ---
        try:
            # Probamos lectura estándar. Si falla por XML, salta al except.
            # sheet_name=0 lee la primera hoja sea cual sea el nombre.
            df_mp = pd.read_excel(path_mp, sheet_name=0)
            leido = True
            print("   [OK] MP leído con motor estándar.")
        except:
            # A veces MP descarga CSVs disfrazados de Excel
            try:
                df_mp = pd.read_csv(path_mp)
                leido = True
                print("   [OK] MP leído como CSV.")
            except:
                print("   [AVISO] Falló lectura estándar MP. Activando Plan B (Excel Real)...")

        # --- INTENTO 2: LECTURA ROBUSTA (XLWINGS - Plan B) ---
        if not leido:
            app = None
            try:
                import xlwings as xw
                app = xw.App(visible=False) 
                wb = app.books.open(path_mp)
                hoja = wb.sheets[0]
                
                # Buscamos la fila de encabezados (MP suele tenerlos en fila 1, pero por si acaso)
                datos_dump = hoja.range('A1:Z5').value 
                header_row = 0
                
                for i, fila in enumerate(datos_dump):
                    fila_str = [str(x).lower() for x in fila if x is not None]
                    # Palabras clave de MP (date_created, fecha de, operation, id)
                    if any("date" in s for s in fila_str) or any("fecha" in s for s in fila_str) or any("operation" in s for s in fila_str):
                        header_row = i + 1 
                        break
                
                if header_row == 0: header_row = 1 
                
                # Leemos la tabla
                df_mp = hoja.range(f'A{header_row}').options(pd.DataFrame, index=False, expand='table').value
                
                wb.close()
                leido = True
                print(f"   [SALVADO] MP recuperado con xlwings (Header fila {header_row}).")
                
            except Exception as ex:
                print(f"   [ERROR CRÍTICO MP] Falló también el Plan B: {ex}")
            finally:
                if app:
                    try: app.quit()
                    except: pass

        # --- PROCESAMIENTO DE COLUMNAS (COMÚN PARA TODOS LOS MÉTODOS) ---
        if not df_mp.empty:
            # Normalizar encabezados a mayúsculas y sin espacios
            df_mp.columns = df_mp.columns.astype(str).str.strip().str.upper()
            
            # Definir nombres de columnas esperadas (MP cambia nombres a veces)
            # Buscamos variantes comunes
            col_monto_mp = "VALOR DEL PRODUCTO (TRANSACTION_AMOUNT)"
            col_fecha_mp = "FECHA DE COMPRA (DATE_CREATED)"
            col_id_mp = "ID DE OPERACIÓN EN MERCADO PAGO" # O OPERATION ID
            
            # Mapeo de columnas si vienen en inglés o variantes
            mapa_cols = {
                'TRANSACTION_AMOUNT': col_monto_mp,
                'DATE_CREATED': col_fecha_mp,
                'OPERATION_ID': 'OPERATION ID (OPERATION_ID)',
                'OPERATION ID': 'OPERATION ID (OPERATION_ID)'
            }
            
            # Intentar renombrar si no encuentra las largas
            for c in df_mp.columns:
                for k, v in mapa_cols.items():
                    if k in c and len(c) < len(v): # Si es una versión corta
                        df_mp.rename(columns={c: v}, inplace=True)

            # Verificar existencia real
            if "OPERATION ID (OPERATION_ID)" in df_mp.columns:
                 op_col = "OPERATION ID (OPERATION_ID)"
            else:
                 op_col = 'ID DE OPERACIÓN EN MERCADO PAGO'

            # 1. Limpieza de Monto
            if col_monto_mp in df_mp.columns:
                # Convertimos a string y reemplazamos caracteres problemáticos
                df_mp[col_monto_mp] = df_mp[col_monto_mp].astype(str).str.replace('$', '', regex=False).str.strip()
                # Si tiene coma y punto (1.000,00), quitamos punto y cambiamos coma
                df_mp[col_monto_mp] = df_mp[col_monto_mp].apply(lambda x: x.replace('.', '').replace(',', '.') if ',' in x and '.' in x else x.replace(',', '.'))
                df_mp[col_monto_mp] = pd.to_numeric(df_mp[col_monto_mp], errors='coerce').fillna(0)
            
            # 2. Limpieza de Fecha
            if col_fecha_mp in df_mp.columns:
                df_mp[col_fecha_mp] = parsear_fecha_mp_iso(df_mp[col_fecha_mp])
                if pd.api.types.is_datetime64_any_dtype(df_mp[col_fecha_mp]): 
                    try: df_mp[col_fecha_mp] = df_mp[col_fecha_mp].dt.tz_localize(None)
                    except: pass

            # 3. Asignación de Turnos
            if not df_t.empty and col_fecha_mp in df_mp.columns:
                df_mp = asignar_turno_desde_excel(df_mp, col_fecha_mp, df_t)
                if 'FECHA_TURNO' in df_mp.columns:
                    df_mp['FECHA_TURNO'] = pd.to_datetime(df_mp['FECHA_TURNO'], dayfirst=True, errors='coerce')
            
            # 4. Columnas Finales Estándar
            df_mp['datetime_col'] = df_mp[col_fecha_mp] if col_fecha_mp in df_mp.columns else pd.NaT
            df_mp['monto_col_numeric'] = df_mp[col_monto_mp] if col_monto_mp in df_mp.columns else 0
            
            df_mp['ID_Original'] = df_mp[op_col].astype(str).str.replace(r'\.0$', '', regex=True) if op_col in df_mp.columns else df_mp.index.astype(str)
            
            df_mp['Estado'] = 'No Conciliado'
            df_mp['Tipo Match'] = ''
            df_mp['Clasificacion'] = '' 
            df_mp['Plataforma'] = 'Mercado Pago'
            df_mp['ID_Cruce'] = ''

    except Exception as e:
        print(f"   [ERROR GENERAL MP]: {e}")
        import traceback; traceback.print_exc()
        df_mp = pd.DataFrame()

   # =========================================================================
   # =========================================================================
    # C. FISERV (VERSIÓN 3: FECHAS BLINDADAS)
    # =========================================================================
    try:
        print("   -> Cargando archivo de FISERV...")
        
        # 1. DETECCIÓN DE ENCABEZADO
        # Leemos las primeras 20 filas para buscar dónde arranca
        try:
            df_temp = pd.read_excel(path_nave, sheet_name=0, header=None, nrows=20)
            header_row = 0
            found_header = False
            for i, row in df_temp.iterrows():
                fila_texto = [str(x).lower().strip() for x in row.values]
                # Buscamos 'fecha' y 'monto' o 'total' en la misma fila
                if any("fecha" in s for s in fila_texto) and (any("monto" in s for s in fila_texto) or any("total" in s for s in fila_texto)):
                    header_row = i
                    found_header = True
                    break
            
            # 2. CARGA REAL
            if not found_header: header_row = 0
            df_nave = pd.read_excel(path_nave, sheet_name=0, header=header_row)

        except Exception as e:
            print(f"   [ERROR LECTURA] Falló búsqueda header: {e}")
            df_nave = pd.read_excel(path_nave, sheet_name=0) 

        # 3. LIMPIEZA DE COLUMNAS
        df_nave.columns = df_nave.columns.astype(str).str.strip()

        # 4. BUSCAR Y PROCESAR FECHA (¡AQUÍ ESTÁ LA CORRECCIÓN!)
        col_fecha_fiserv = None
        posibles_nombres_fecha = ['Fecha y hora', 'Fecha', 'Fecha Operación', 'Fecha de operación', 'Date']
        
        for nombre in posibles_nombres_fecha:
            if nombre in df_nave.columns:
                col_fecha_fiserv = nombre
                break
        
        if col_fecha_fiserv:
            print(f"   [DEBUG] Columna fecha detectada: '{col_fecha_fiserv}'")
            # --- IMPRIMIR MUESTRA PARA DIAGNÓSTICO ---
            print(f"   [DEBUG] Muestra cruda: {df_nave[col_fecha_fiserv].head(3).tolist()}")

            # A) Convertir todo a string para limpiar
            df_nave['temp_date'] = df_nave[col_fecha_fiserv].astype(str).str.strip()
            
            # B) Limpiezas comunes que rompen la fecha
            # Quitar '.0' al final si vino como número, quitar 'T', quitar 'hs'
            df_nave['temp_date'] = df_nave['temp_date'].str.replace(r'\.0$', '', regex=True)
            df_nave['temp_date'] = df_nave['temp_date'].str.replace('T', ' ', regex=False)
            df_nave['temp_date'] = df_nave['temp_date'].str.replace(' hs', '', regex=False)
            df_nave['temp_date'] = df_nave['temp_date'].str.replace(' hrs', '', regex=False)
            df_nave['temp_date'] = df_nave['temp_date'].str.replace('nan', '', regex=False, case=False)
            df_nave['temp_date'] = df_nave['temp_date'].str.replace('NaT', '', regex=False, case=False)

            # C) Intentar conversión (Primero dia/mes, luego formato estándar)
            df_nave['datetime_col'] = pd.to_datetime(df_nave['temp_date'], dayfirst=True, errors='coerce')
            
            # Si falló más del 90% de las fechas, intentamos sin dayfirst (formato americano o ISO)
            if df_nave['datetime_col'].isna().sum() > (len(df_nave) * 0.9) and len(df_nave) > 0:
                 print("   [AVISO] Falló formato DD/MM. Reintentando formato ISO/Americano...")
                 df_nave['datetime_col'] = pd.to_datetime(df_nave['temp_date'], errors='coerce')

            # Borrar temporal
            df_nave.drop(columns=['temp_date'], inplace=True, errors='ignore')
        else:
            print(f"   [ALERTA CRÍTICA] No se encontró columna de Fecha. Buscado: {posibles_nombres_fecha}")
            print(f"   Columnas disponibles: {df_nave.columns.tolist()}")
            df_nave['datetime_col'] = pd.NaT

        # 5. BUSCAR MONTO
        col_monto_fiserv = None
        posibles_nombres_monto = ['Monto total', 'Monto', 'Importe', 'Total', 'Amount']
        for nombre in posibles_nombres_monto:
            if nombre in df_nave.columns:
                col_monto_fiserv = nombre
                break

        if col_monto_fiserv:
            # Limpieza para que '1.000,50' sea número
            df_nave[col_monto_fiserv] = df_nave[col_monto_fiserv].astype(str).str.replace('$', '', regex=False).str.replace('ARS', '', regex=False).str.strip()
            df_nave['monto_col_numeric'] = df_nave[col_monto_fiserv].apply(
                lambda x: float(x.replace('.', '').replace(',', '.')) if ',' in x and '.' in x else 
                          (float(x.replace(',', '.')) if ',' in x else pd.to_numeric(x, errors='coerce'))
            ).fillna(0)
        else:
            print("   [ALERTA] No se encontró columna de Monto.")
            df_nave['monto_col_numeric'] = 0

        # 6. ASIGNACIÓN DE TURNO
        if not df_t.empty: 
            df_nave = asignar_turno_desde_excel(df_nave, 'datetime_col', df_t)
            if 'FECHA_TURNO' in df_nave.columns:
                df_nave['FECHA_TURNO'] = pd.to_datetime(df_nave['FECHA_TURNO'], dayfirst=True, errors='coerce')

        # 7. ID ORIGINAL
        col_id_fiserv = None
        posibles_ids = ['Nro. Transacción', 'Nro Transaccion', 'Referencia', 'Reference', 'ID']
        for nombre in posibles_ids:
            if nombre in df_nave.columns:
                col_id_fiserv = nombre
                break
        
        if col_id_fiserv:
            df_nave['ID_Original'] = df_nave[col_id_fiserv].astype(str).str.replace(r'\.0$', '', regex=True)
        else:
            df_nave['ID_Original'] = df_nave.index.astype(str)

        # 8. ESTANDARIZACIÓN
        df_nave['Estado'] = 'No Conciliado'
        df_nave['Tipo Match'] = ''
        df_nave['Clasificacion'] = ''
        df_nave['Plataforma'] = 'Fiserv'
        df_nave['ID_Cruce'] = ''
        
        # Columna visual (Chequeamos si datetime_col tiene datos válidos)
        df_nave['Fecha'] = df_nave['datetime_col'].dt.strftime('%d/%m/%Y')

        # 9. LIMPIEZA FINAL DE COLUMNAS
        cols_mantener = ['datetime_col', 'monto_col_numeric', 'ID_Original', 'TURNO', 'FECHA_TURNO', 'Estado', 'Tipo Match', 'Clasificacion', 'Plataforma', 'ID_Cruce', 'Fecha']
        cols_finales = [c for c in df_nave.columns if c in cols_mantener or c == col_fecha_fiserv or c == col_monto_fiserv]
        df_nave = df_nave[cols_finales].copy()

        print(f"   [OK] Fiserv cargado correctamente: {len(df_nave)} registros.")
        
    except Exception as e:
        print(f"   [ERROR] Cargando Fiserv: {e}")
        import traceback
        traceback.print_exc()
        df_nave = pd.DataFrame()


    # =========================================================================
    # D. SISTEMA
    # =========================================================================
    try:
        df_sys = pd.read_excel(path_ventas)
        df_sys = df_sys[~df_sys['COMPROBANT'].astype(str).str.contains("TOTALES|FECHA:", na=False)]
        
        # --- [CORRECCIÓN INICIO]: RELLENAR DATOS HACIA ABAJO PARA PAGOS DIVIDIDOS ---
        # Definimos qué columnas son las que se deben repetir si están vacías en la fila de abajo
        cols_rellenar = ['FECHA', 'TURNO', 'HORA', 'COMPROBANT']
        
        for col in cols_rellenar:
            if col in df_sys.columns:
                # 1. Convertimos espacios vacíos o texto vacío a NaN (Nulo real)
                df_sys[col] = df_sys[col].replace(r'^\s*$', np.nan, regex=True)
                
                # 2. Hacemos "Forward Fill" (Rellenar con el valor de arriba)
                df_sys[col] = df_sys[col].ffill()
        # --- [CORRECCIÓN FIN] ---

        # 1. Normalizar turno Y GUARDAR EL ORIGINAL
        df_sys['TURNO'] = df_sys['TURNO'].astype(str).str.upper().str.strip().str.replace("MAÑ", "MAÑANA").str.replace("TAR", "TARDE")
        
        # --- RESGUARDAR EL TURNO ORIGINAL ---
        df_sys['Turno_Original_Archivo'] = df_sys['TURNO'].copy()
        # -------------------------------------------

        # 2. Convertir Fechas
        df_sys['FECHA'] = pd.to_datetime(df_sys['FECHA'], dayfirst=True, errors='coerce')
        df_sys['Fecha_Original_Archivo'] = df_sys['FECHA'].dt.date
        
        print("    -> Procesando fechas del Sistema (con lógica 6 AM)...")
        
        # 3. Combinar fecha y hora (AHORA LA FILA 2 YA TENDRÁ LA HORA 15:49 GRACIAS AL FFILL)
        df_sys['Fecha y hora '] = df_sys.apply(combinar_fecha_hora_sistema, axis=1)
        
        # 4. Asignar turno REAL (Sobrescribe la columna TURNO con la verdad del horario)
        df_sys['datetime_col'] = df_sys['Fecha y hora ']
        
        if not df_t.empty:
            df_sys = asignar_turno_desde_excel(df_sys, 'datetime_col', df_t)
            
            # --- NUEVO: COMPARAR Y GENERAR MOTIVO ---
            def diagnosticar_cambio_turno(row):
                orig = str(row['Turno_Original_Archivo']).strip()
                real = str(row['TURNO']).strip()
                
                # Caso 1: Quedó fuera de rango horario
                if real == "Fuera de turno":
                    return "ALERTA: Hora no coincide con ningún turno activo"
                
                # Caso 2: El archivo decía una cosa, pero el horario dice otra
                # (Ignoramos si son iguales)
                if orig != real and orig != 'nan' and orig != 'NAT':
                    return f"CORREGIDO: Cajero puso '{orig}'"
                
                return "" # Si coinciden, lo dejamos limpio
            
            df_sys['Motivo_Cambio_Turno'] = df_sys.apply(diagnosticar_cambio_turno, axis=1)
            # ----------------------------------------

            # Normalizar FECHA_TURNO
            if 'FECHA_TURNO' in df_sys.columns:
                df_sys['FECHA_TURNO'] = pd.to_datetime(df_sys['FECHA_TURNO'], dayfirst=True, errors='coerce')
                df_sys['FECHA'] = df_sys['FECHA_TURNO'].dt.date
        else:
            df_sys['TURNO'] = "Sin Data Turnos"
            df_sys['FECHA_TURNO'] = pd.NaT
            df_sys['FECHA'] = df_sys['Fecha y hora '].dt.date
            df_sys['Motivo_Cambio_Turno'] = "No hay archivo maestro de turnos"

        df_sys['monto_col_numeric'] = pd.to_numeric(df_sys['IMPORTE'], errors='coerce')
        df_sys['ID_Original'] = df_sys.index.astype(str)
        df_sys['Estado'] = 'No Conciliado'
        df_sys['Tipo Match'] = ''
        df_sys['Plataforma'] = 'Sistema'
        df_sys['ID_Cruce'] = '' 
        
        df_sys['Medio_Cobro_Norm'] = df_sys['COBRO'].astype(str).str.replace(' ', '').str.upper().str.strip()
        df_sys['Plataforma_Real'] = df_sys['Medio_Cobro_Norm'].apply(identificar_plataforma_real)
    except Exception as e: 
        print(f"Error Sistema: {e}")
        import traceback
        traceback.print_exc()
        df_sys = pd.DataFrame()

    print("   -> Cargando Adicionales...")
    df_adic = procesar_archivo_adicion(path_adicion)

    print(">>> Carga finalizada.")

    # =========================================================================
    # E. DESCUENTOS (Nuevo)
    # =========================================================================
    print("    -> Cargando Descuentos...")
    try:
        df_desc = pd.read_excel(path_descuento, sheet_name=0)
    except Exception as e:
        print(f"    [ADVERTENCIA] No se pudo cargar descuentos: {e}")
        df_desc = pd.DataFrame()

    print(">>> Carga finalizada.")

    df_audit = procesar_archivo_auditoria(path_auditoria)
    
    

    # Retornamos 8 variables (agregamos df_banco al final)
    return df_mp, df_nave, df_sys, df_t, df_adic, df_desc, df_audit
# =============================================================================
# 4. MEMORIA DE CLASIFICACIÓN (RECUPERAR DATOS)
# =============================================================================

def recuperar_clasificaciones(df_actual, ruta_archivo_previo, hoja_origen):
    if not os.path.exists(ruta_archivo_previo): return df_actual
    
    print(f"   -> Buscando clasificaciones previas en {hoja_origen}...")
    try:
        df_prev = pd.read_excel(ruta_archivo_previo, sheet_name=hoja_origen)
        if 'ID_Original' in df_prev.columns and 'Clasificacion' in df_prev.columns:
            # Crear diccionario ID -> Clasificacion
            df_prev['ID_Original'] = df_prev['ID_Original'].astype(str).str.replace(r'\.0$', '', regex=True)
            mapa = df_prev.set_index('ID_Original')['Clasificacion'].dropna().to_dict()
            
            # Mapear al actual
            df_actual['Clasificacion'] = df_actual['ID_Original'].map(mapa).fillna(df_actual['Clasificacion'])
            print(f"      Recuperadas {len(mapa)} etiquetas.")
    except Exception as e:
        print(f"      No se pudo recuperar memoria: {e}")
    return df_actual

# =============================================================================
# 5. MOTOR DE CONCILIACIÓN (Misma lógica del original)
# =============================================================================

def correr_conciliacion(df_plat, df_sys, nombre_fase, target_medio, ignorar_medio=False, solo_nave=False):
    print(f"\n--- Fase: {nombre_fase} ---")
    
    # Filtro inicial por medio de pago
    if target_medio and not ignorar_medio:
        df_sys_subset = df_sys[df_sys['Medio_Cobro_Norm'] == target_medio]
    else:
        df_sys_subset = df_sys
    
    # *** NUEVO: Si es fase global, SOLO buscar en registros de fiserv ***
    if solo_nave and 'Plataforma_Real' in df_sys_subset.columns:
        cantidad_antes = len(df_sys_subset)
        df_sys_subset = df_sys_subset[df_sys_subset['Plataforma_Real'] == 'Fiserv']
        cantidad_incluida = len(df_sys_subset)
        if cantidad_incluida > 0:
            print(f"   [INFO] Buscando SOLO en {cantidad_incluida} registros de FISERV (detectar descalces).")

    for regla, min_tol, money_tol in REGLAS_CONCILIACION:
        matches = 0
        for i_p in df_plat.index:
            if df_plat.at[i_p, 'Estado'] != 'No Conciliado': 
                continue
            
            t_p = df_plat.at[i_p, 'datetime_col']
            m_p = df_plat.at[i_p, 'monto_col_numeric']
            turno_p = df_plat.at[i_p, 'TURNO']
            
            if pd.isna(t_p) or pd.isna(m_p): 
                continue

            # Filtros: Mismo Turno, No Conciliado
            mask = (
                (df_sys_subset['TURNO'] == turno_p) & 
                (df_sys_subset['Estado'] == 'No Conciliado')
            )
            
            # Filtro por FECHA_TURNO
            if 'FECHA_TURNO' in df_plat.columns and 'FECHA_TURNO' in df_sys_subset.columns:
                fecha_turno_p = df_plat.at[i_p, 'FECHA_TURNO']
                if pd.notna(fecha_turno_p):
                    mask = mask & (df_sys_subset['FECHA_TURNO'] == fecha_turno_p)
            
            candidatos = df_sys_subset[mask]
            
            # Filtro Dinero
            candidatos = candidatos[candidatos['monto_col_numeric'].between(m_p - money_tol, m_p + money_tol)]
            
            # Filtro Tiempo
            match = pd.DataFrame()
            if regla.startswith("R3"): 
                match = candidatos[candidatos['datetime_col'].dt.date == t_p.date()]
            else:
                delta = pd.Timedelta(minutes=min_tol)
                match = candidatos[candidatos['datetime_col'].between(t_p - delta, t_p + delta)]
            
            if not match.empty:
                i_s = match.index[0]
                
                # Marcar Match
                tipo_match = f"{nombre_fase} ({regla})"
                df_plat.at[i_p, 'Estado'] = 'Conciliado'
                df_plat.at[i_p, 'Tipo Match'] = tipo_match
                df_plat.at[i_p, 'ID_Cruce'] = df_sys.at[i_s, 'ID_Original']
                
                df_sys.at[i_s, 'Estado'] = 'Conciliado'
                df_sys.at[i_s, 'Tipo Match'] = tipo_match
                df_sys.at[i_s, 'ID_Cruce'] = df_plat.at[i_p, 'ID_Original']
                
                matches += 1
        
        if matches > 0: 
            print(f"   -> {matches} matches con {regla}")

def correr_conciliacion_inter_turnos(df_plat, df_sys, target_medio):
    print(f"\n--- Fase Especial: CRUCE INTER-TURNOS (Rescate de 3hs) ---")
    
    tolerancia_min = 180 
    money_tol = 5 

    if isinstance(target_medio, list):
        df_sys_subset = df_sys[df_sys['Medio_Cobro_Norm'].isin(target_medio)]
    elif isinstance(target_medio, str):
        df_sys_subset = df_sys[df_sys['Medio_Cobro_Norm'] == target_medio]
    else:
        df_sys_subset = df_sys

    matches = 0
    
    for i_p in df_plat.index:
        if df_plat.at[i_p, 'Estado'] != 'No Conciliado': 
            continue
        
        t_p = df_plat.at[i_p, 'datetime_col']
        m_p = df_plat.at[i_p, 'monto_col_numeric']
        turno_real = df_plat.at[i_p, 'TURNO'] 

        # --- REGLA: Si la plataforma está fuera de turno, NO TOCAR ---
        if pd.isna(turno_real) or turno_real == "Fuera de turno":
            continue

        if pd.isna(t_p) or pd.isna(m_p): 
            continue

        mask = (df_sys_subset['Estado'] == 'No Conciliado')
        
        # *** FILTRO POR FECHA_TURNO (solo si existe la columna en ambos) ***
        if 'FECHA_TURNO' in df_plat.columns and 'FECHA_TURNO' in df_sys_subset.columns:
            fecha_turno_p = df_plat.at[i_p, 'FECHA_TURNO']
            if pd.notna(fecha_turno_p):
                mask = mask & (df_sys_subset['FECHA_TURNO'] == fecha_turno_p)
        
        candidatos = df_sys_subset[mask]
        candidatos = candidatos[candidatos['monto_col_numeric'].between(m_p - money_tol, m_p + money_tol)]
        
        delta = pd.Timedelta(minutes=tolerancia_min)
        match = candidatos[candidatos['datetime_col'].between(t_p - delta, t_p + delta)]
        
        if not match.empty:
            i_s = match.index[0]
            tipo_match = "Recupero Inter-Turno"
            
            turno_viejo = df_sys.at[i_s, 'TURNO']
            df_sys.at[i_s, 'TURNO'] = turno_real 
            
            nota = f"Corregido: {turno_viejo} -> {turno_real}"
            df_sys.at[i_s, 'Clasificacion'] = nota
            df_plat.at[i_p, 'Clasificacion'] = nota

            df_plat.at[i_p, 'Estado'] = 'Conciliado'
            df_plat.at[i_p, 'Tipo Match'] = tipo_match
            df_plat.at[i_p, 'ID_Cruce'] = df_sys.at[i_s, 'ID_Original']
            
            df_sys.at[i_s, 'Estado'] = 'Conciliado'
            df_sys.at[i_s, 'Tipo Match'] = tipo_match
            df_sys.at[i_s, 'ID_Cruce'] = df_plat.at[i_p, 'ID_Original']
            
            matches += 1
            print(f"   [RESCATE] ID {df_sys.at[i_s, 'ID_Original']}: Movido de {turno_viejo} a {turno_real}")

    if matches > 0: 
        print(f"   -> ¡Se recuperaron {matches} ventas cruzadas de turno!")

def correr_conciliacion_nave_invertida(df_nave, df_sys):
    """
    Busca transacciones de FISERV que coincidan con registros del sistema 
    que NO están marcados como Fiserv (posible error de clasificación).
    """
    # CAMBIO 1: Texto de consola corregido
    print(f"\n--- Fase: Fiserv Global → Registros NO-FISERV ---")
    print("    Buscando transacciones FISERV que se registraron con otro medio...")
    
    # Filtrar SOLO registros que NO son FISERV
    if 'Plataforma_Real' in df_sys.columns:
        df_sys_subset = df_sys[df_sys['Plataforma_Real'] != 'Fiserv']
        print(f"   [INFO] Buscando en {len(df_sys_subset)} registros NO-FISERV.")
    else:
        df_sys_subset = df_sys

    for regla, min_tol, money_tol in REGLAS_CONCILIACION:
        matches = 0
        for i_n in df_nave.index:
            if df_nave.at[i_n, 'Estado'] != 'No Conciliado': 
                continue
            
            t_n = df_nave.at[i_n, 'datetime_col']
            m_n = df_nave.at[i_n, 'monto_col_numeric']
            turno_n = df_nave.at[i_n, 'TURNO']
            
            if pd.isna(t_n) or pd.isna(m_n): 
                continue

            mask = (
                (df_sys_subset['TURNO'] == turno_n) & 
                (df_sys_subset['Estado'] == 'No Conciliado')
            )
            
            if 'FECHA_TURNO' in df_nave.columns and 'FECHA_TURNO' in df_sys_subset.columns:
                fecha_turno_n = df_nave.at[i_n, 'FECHA_TURNO']
                if pd.notna(fecha_turno_n):
                    mask = mask & (df_sys_subset['FECHA_TURNO'] == fecha_turno_n)
            
            candidatos = df_sys_subset[mask]
            candidatos = candidatos[candidatos['monto_col_numeric'].between(m_n - money_tol, m_n + money_tol)]
            
            match = pd.DataFrame()
            if regla.startswith("R3"): 
                match = candidatos[candidatos['datetime_col'].dt.date == t_n.date()]
            else:
                delta = pd.Timedelta(minutes=min_tol)
                match = candidatos[candidatos['datetime_col'].between(t_n - delta, t_n + delta)]
            
            if not match.empty:
                i_s = match.index[0]
                
                # CAMBIO 2: Esto es lo que va al Excel. Ahora dirá Fiserv.
                tipo_match = f"⚠️ DESCALCE: Fiserv → {df_sys.at[i_s, 'COBRO']} ({regla})"
                
                df_nave.at[i_n, 'Estado'] = 'Conciliado'
                df_nave.at[i_n, 'Tipo Match'] = tipo_match
                df_nave.at[i_n, 'ID_Cruce'] = df_sys.at[i_s, 'ID_Original']
                
                df_sys.at[i_s, 'Estado'] = 'Conciliado'
                df_sys.at[i_s, 'Tipo Match'] = tipo_match
                df_sys.at[i_s, 'ID_Cruce'] = df_nave.at[i_n, 'ID_Original']
                
                matches += 1
        
        if matches > 0: 
            print(f"   -> ⚠️ {matches} DESCALCES detectados con {regla}")




# =============================================================================
# 6. REPORTES (Formato idéntico al solicitado)
# =============================================================================

def generar_reporte_plano(writer, df_mp, df_nave, df_sys):
    print("Generando Reporte Estadístico Plano...")
    
    # Columnas requeridas
    cols_req = ['datetime_col', 'TURNO', 'Plataforma', 'monto_col_numeric', 'Estado']
    
    # --- FUNCIÓN AUXILIAR PARA EVITAR ERRORES SI EL ARCHIVO ESTÁ VACÍO ---
    def obtener_datos_seguros(df_origen, columnas):
        if df_origen.empty:
            return pd.DataFrame(columns=columnas)
        
        # Verificar si falta alguna columna
        faltantes = [c for c in columnas if c not in df_origen.columns]
        if faltantes:
            # Si faltan columnas, devolvemos vacío para no romper el programa
            return pd.DataFrame(columns=columnas)
            
        return df_origen[columnas].copy()

    # 1. Obtener datos de forma segura
    df1 = obtener_datos_seguros(df_mp, cols_req)
    df2 = obtener_datos_seguros(df_nave, cols_req)
    
    # 2. Preparar datos del Sistema (caso especial porque renombramos columna)
    if not df_sys.empty and 'COBRO' in df_sys.columns:
        df3 = df_sys.copy()
        df3['Plataforma'] = df_sys['COBRO']
        
        # Aseguramos que existan todas las columnas requeridas
        for col in cols_req:
            if col not in df3.columns:
                df3[col] = 0 # O valor por defecto
        
        df3 = df3[cols_req]
    else:
        df3 = pd.DataFrame(columns=cols_req)
    
    # 3. Unir todo
    df_total = pd.concat([df1, df2, df3], ignore_index=True)
    
    if df_total.empty:
        # Si todo está vacío, crear hoja vacía y salir
        pd.DataFrame(columns=['Mensaje']).to_excel(writer, sheet_name='Reporte_Estadistico_Plano', index=False)
        return

    df_total['Fecha'] = df_total['datetime_col'].dt.strftime('%d/%m/%Y')
    df_total['Cantidad'] = 1
    
    # Pivot Table
    df_agg = df_total.groupby(['Fecha', 'TURNO', 'Plataforma', 'Estado']).agg(
        Monto=('monto_col_numeric', 'sum'),
        Cant=('Cantidad', 'count')
    ).reset_index()
    
    pivot_monto = df_agg.pivot_table(index=['Fecha', 'TURNO', 'Plataforma'], columns='Estado', values='Monto').fillna(0)
    pivot_cant = df_agg.pivot_table(index=['Fecha', 'TURNO', 'Plataforma'], columns='Estado', values='Cant').fillna(0)
    
    pivot_monto.columns = [f"Monto {c}" for c in pivot_monto.columns]
    pivot_cant.columns = [f"Cant {c}" for c in pivot_cant.columns]
    
    final = pivot_monto.join(pivot_cant).reset_index()
    
    # Totales
    if 'Monto Conciliado' not in final.columns: final['Monto Conciliado'] = 0
    if 'Monto No Conciliado' not in final.columns: final['Monto No Conciliado'] = 0
    final['Total Monto'] = final['Monto Conciliado'] + final['Monto No Conciliado']
    
    final.to_excel(writer, sheet_name='Reporte_Estadistico_Plano', index=False)

def generar_auditoria(writer, df_mp, df_nave, df_sys):
    print("Generando Auditoría de Pendientes...")
    
    # Filtrar solo pendientes y que NO sean excluidos (Interno, etc)
    mask_excl_mp = calcular_mascara_exclusion(df_mp['Clasificacion'])
    pend_mp = df_mp[(df_mp['Estado'] == 'No Conciliado') & (~mask_excl_mp)].copy()
    
    mask_excl_nav = calcular_mascara_exclusion(df_nave['Clasificacion'])
    pend_nav = df_nave[(df_nave['Estado'] == 'No Conciliado') & (~mask_excl_nav)].copy()
    
    pend_sys = df_sys[df_sys['Estado'] == 'No Conciliado'].copy()
    
    # Formatear para reporte
    res = []
    
    # 1. MERCADO PAGO
    if not pend_mp.empty:
        temp = pd.DataFrame()
        temp['Fecha y Hora'] = pend_mp['datetime_col']
        temp['Fecha Turno'] = pend_mp['FECHA_TURNO'] if 'FECHA_TURNO' in pend_mp.columns else pd.NaT
        temp['TURNO'] = pend_mp['TURNO']
        temp['Origen'] = 'Mercado Pago'
        temp['ID'] = pend_mp['ID_Original']
        temp['Monto'] = pend_mp['monto_col_numeric']
        temp['Clasificacion'] = pend_mp['Clasificacion']
        temp['Plataforma Real'] = ''  # MP no necesita esta columna
        res.append(temp)
    
    # 2. FISERV
    if not pend_nav.empty:
        temp = pd.DataFrame()
        temp['Fecha y Hora'] = pend_nav['datetime_col']
        temp['Fecha Turno'] = pend_nav['FECHA_TURNO'] if 'FECHA_TURNO' in pend_nav.columns else pd.NaT
        temp['TURNO'] = pend_nav['TURNO']
        temp['Origen'] = 'Fiserv'
        temp['ID'] = pend_nav['ID_Original']
        temp['Monto'] = pend_nav['monto_col_numeric']
        temp['Clasificacion'] = pend_nav['Clasificacion']
        temp['Plataforma Real'] = ''  # NAVE no necesita esta columna
        res.append(temp)
    
    # 3. SISTEMA (con Plataforma_Real)
    if not pend_sys.empty:
        temp = pd.DataFrame()
        temp['Fecha y Hora'] = pend_sys['datetime_col']
        temp['Fecha Turno'] = pend_sys['FECHA_TURNO'] if 'FECHA_TURNO' in pend_sys.columns else pd.NaT
        temp['TURNO'] = pend_sys['TURNO']
        temp['Origen'] = 'Sistema (' + pend_sys['COBRO'].astype(str) + ')'
        temp['ID'] = pend_sys['ID_Original']
        temp['Monto'] = pend_sys['monto_col_numeric']
        temp['Clasificacion'] = pend_sys['Clasificacion'] if 'Clasificacion' in pend_sys.columns else ''
        temp['Plataforma Real'] = pend_sys['Plataforma_Real'] if 'Plataforma_Real' in pend_sys.columns else ''
        res.append(temp)
        
    if res:
        full_audit = pd.concat(res, ignore_index=True)
        # Ordenar por Fecha Turno y Turno
        if 'Fecha Turno' in full_audit.columns:
            full_audit = full_audit.sort_values(['Fecha Turno', 'TURNO', 'Fecha y Hora'])
        full_audit.to_excel(writer, sheet_name='Auditoría_Pendientes', index=False)
    else:
        pd.DataFrame({'Info': ['Todo conciliado']}).to_excel(writer, sheet_name='Auditoría_Pendientes')

def exportar_al_macro(df_mp, df_nave, df_sys, df_adic, df_desc, df_audit):
    """
    Exporta al archivo macro CON PROTECCIÓN TOTAL DE FÓRMULAS.
    Busca las hojas ignorando mayúsculas/espacios para evitar borrarlas.
    """
    print("\n>>> Generando exportación al MACRO...")
    
    if not os.path.exists(path_macro):
        print(f"    [ERROR] No se encontró el archivo macro en: {path_macro}")
        return
    
    try:
        import xlwings as xw
    except ImportError:
        print("    [ERROR] xlwings no está instalado. Ejecutá: pip install xlwings")
        return
    
    app = xw.App(visible=False)
    try:
        wb = app.books.open(path_macro)
        
        # --- FUNCIÓN BLINDADA PARA OBTENER HOJAS ---
        def obtener_hoja_segura(wb, nombre_deseado):
            # 1. Normalizamos el nombre buscado (mayúsculas y sin espacios)
            target = str(nombre_deseado).strip().upper()
            
            # 2. Buscamos en todas las hojas existentes
            for sheet in wb.sheets:
                if sheet.name.strip().upper() == target:
                    try:
                        sheet.cells.clear() 
                    except:
                        pass # Si falla limpiar (hoja protegida), seguimos
                    return sheet
            
            # 3. Si terminó el bucle y no la encontró, recién ahí la crea
            print(f"    [AVISO] Creando hoja nueva: {nombre_deseado}")
            return wb.sheets.add(nombre_deseado)

        # =====================================================================
        # 1. REPORTE ESTADÍSTICO
        # =====================================================================
        print("    -> Generando Reporte Estadístico...")
        
        cols_req = ['datetime_col', 'TURNO', 'Plataforma', 'monto_col_numeric', 'Estado']
        
        # Preparar datos seguros
        def get_safe(df, cols): 
            return df[cols].copy() if not df.empty and all(c in df.columns for c in cols) else pd.DataFrame(columns=cols)

        df1 = get_safe(df_mp, cols_req)
        df2 = get_safe(df_nave, cols_req)
        
        if not df_sys.empty and 'COBRO' in df_sys.columns:
            df3 = df_sys.copy()
            df3['Plataforma'] = df_sys['COBRO']
            for c in cols_req: 
                if c not in df3.columns: df3[c] = 0
            df3 = df3[cols_req]
        else:
            df3 = pd.DataFrame(columns=cols_req)
        
        df_total = pd.concat([df1, df2, df3], ignore_index=True)
        
        if not df_total.empty:
            df_total['Fecha'] = df_total['datetime_col'].dt.strftime('%d/%m/%Y')
            df_total['Cantidad'] = 1
            
            df_agg = df_total.groupby(['Fecha', 'TURNO', 'Plataforma', 'Estado']).agg(
                Monto=('monto_col_numeric', 'sum'),
                Cant=('Cantidad', 'count')
            ).reset_index()
            
            pivot_monto = df_agg.pivot_table(index=['Fecha', 'TURNO', 'Plataforma'], columns='Estado', values='Monto').fillna(0)
            pivot_cant = df_agg.pivot_table(index=['Fecha', 'TURNO', 'Plataforma'], columns='Estado', values='Cant').fillna(0)
            
            pivot_monto.columns = [f"Monto {c}" for c in pivot_monto.columns]
            pivot_cant.columns = [f"Cant {c}" for c in pivot_cant.columns]
            
            df_reporte = pivot_monto.join(pivot_cant).reset_index()
            
            if 'Monto Conciliado' not in df_reporte.columns: df_reporte['Monto Conciliado'] = 0
            if 'Monto No Conciliado' not in df_reporte.columns: df_reporte['Monto No Conciliado'] = 0
            df_reporte['Total Monto'] = df_reporte['Monto Conciliado'] + df_reporte['Monto No Conciliado']
            
            ws_reporte = obtener_hoja_segura(wb, 'Reporte Estadistico')
            ws_reporte.range('A1').options(index=False).value = df_reporte
            ws_reporte.autofit()
            print("    -> Reporte Estadístico exportado.")

        # =====================================================================
        # 2. NO CONCILIADO (Y CRUCES DE MEDIOS)
        # =====================================================================
        print("    -> Generando tablas de No Conciliado...")
        
        def preparar_tabla(df_source, col_id_primary, titulo):
            if df_source.empty:
                return pd.DataFrame(columns=['ID', 'Fecha y Hora', 'Fecha Turno', 'TURNO', 'Monto', 'Detalle']), titulo
            
            temp = pd.DataFrame()
            temp['ID'] = df_source[col_id_primary] if col_id_primary in df_source.columns else df_source['ID_Original']
            temp['Fecha y Hora'] = df_source['datetime_col']
            temp['Fecha Turno'] = df_source['FECHA_TURNO'] if 'FECHA_TURNO' in df_source.columns else pd.NaT
            temp['TURNO'] = df_source['TURNO']
            temp['Monto'] = df_source['monto_col_numeric']
            
            if 'Tipo Match' in df_source.columns and str(df_source['Tipo Match'].iloc[0]) != '':
                 temp['Detalle'] = df_source['Tipo Match']
            elif 'Plataforma_Real' in df_source.columns:
                 temp['Detalle'] = df_source['Plataforma_Real']
            else:
                 temp['Detalle'] = ''
            
            return temp, titulo
        
        # Filtros
        mask_excl_mp = calcular_mascara_exclusion(df_mp['Clasificacion']) if not df_mp.empty else False
        df_mp_no_conc = df_mp[(df_mp['Estado'] == 'No Conciliado') & (~mask_excl_mp)] if not df_mp.empty else pd.DataFrame()
        
        mask_excl_nav = calcular_mascara_exclusion(df_nave['Clasificacion']) if not df_nave.empty else False
        df_nave_no_conc = df_nave[(df_nave['Estado'] == 'No Conciliado') & (~mask_excl_nav)] if not df_nave.empty else pd.DataFrame()
        
        # Preparar tablas
        t1, h1 = preparar_tabla(df_mp_no_conc, 'Operation ID (operation_id)', "MP NO CONCILIADO")
        t2, h2 = preparar_tabla(df_nave_no_conc, 'ID_Original', "FISERV NO CONCILIADO") 
        
        df_sys_mp = df_sys[(df_sys['Medio_Cobro_Norm'].isin(['QRMERC.PAGO', 'MERCADOPAGO'])) & (df_sys['Estado'] == 'No Conciliado')] if not df_sys.empty else pd.DataFrame()
        t3, h3 = preparar_tabla(df_sys_mp, 'ID_Original', "SISTEMA (MP/QR) NO CONCILIADO")
        
        df_sys_nave = df_sys[(df_sys['Plataforma_Real'] == 'Fiserv') & (df_sys['Estado'] == 'No Conciliado')] if not df_sys.empty and 'Plataforma_Real' in df_sys.columns else pd.DataFrame()
        t4, h4 = preparar_tabla(df_sys_nave, 'ID_Original', "SISTEMA (FISERV) NO CONCILIADO")
        
        df_sys_cc = df_sys[(df_sys['Medio_Cobro_Norm'] == TARGET_CTACTE) & (df_sys['Estado'] == 'Conciliado')] if not df_sys.empty else pd.DataFrame()
        t_cc, h_cc = preparar_tabla(df_sys_cc, 'ID_Original', "✅ AUDITORÍA CTA CTE (Recuperadas/Conciliadas)")

        match_rescate = "Rescate" # Palabra clave que pusimos en el conciliador
        
        # 1. Buscamos en MP
        rescates_mp = pd.DataFrame()
        if not df_mp.empty:
            rescates_mp = df_mp[df_mp['Tipo Match'].astype(str).str.contains(match_rescate, case=False, na=False)].copy()
            if not rescates_mp.empty: rescates_mp['Origen'] = 'Mercado Pago'

        # 2. Buscamos en NAVE
        rescates_nav = pd.DataFrame()
        if not df_nave.empty:
            rescates_nav = df_nave[df_nave['Tipo Match'].astype(str).str.contains(match_rescate, case=False, na=False)].copy()
            if not rescates_nav.empty: rescates_nav['Origen'] = 'Fiserv'

        # 3. Unimos
        df_rescates = pd.concat([rescates_mp, rescates_nav], ignore_index=True)
        
        # 4. Preparamos la tabla visual
        t7, h7 = preparar_tabla(df_rescates, 'ID_Original', "✅ RECUPERADO (Marcado erróneamente en Caja)")
        
        # (Opcional) Agregamos columna para saber si vino de MP o Nave
        if not t7.empty and 'Origen' in df_rescates.columns:
             t7.insert(0, 'Plat.', df_rescates['Origen'].values)
        # ---------------------------------------------------------
        
        # Cruces
        cruce_mp = df_mp[df_mp['Tipo Match'].astype(str).str.contains("Registros NAVE", na=False)].copy() if not df_mp.empty else pd.DataFrame()
        cruce_nav = df_nave[df_nave['Tipo Match'].astype(str).str.contains("DESCALCE", na=False)].copy() if not df_nave.empty else pd.DataFrame()
        df_cruces = pd.concat([cruce_mp, cruce_nav])
        t6, h6 = preparar_tabla(df_cruces, 'ID_Original', "⚠️ CRUCE DE MEDIOS (ADVERTENCIA)")
        
        # Volcado
        ws_no_conc = obtener_hoja_segura(wb, 'No conciliado')
        
        tablas = [(t1, h1), (t2, h2), (t3, h3), (t4, h4), (t_cc, h_cc), (t6, h6), (t7, h7)]
        current_col = 1
        for df_tabla, titulo in tablas:
            ws_no_conc.range((1, current_col)).value = titulo
            try:
                ws_no_conc.range((1, current_col)).api.Font.Bold = True
                if "CRUCE" in titulo: ws_no_conc.range((1, current_col)).api.Font.Color = (255, 0, 0)
            except: pass
            
            if not df_tabla.empty:
                ws_no_conc.range((2, current_col)).options(index=False).value = df_tabla
                current_col += len(df_tabla.columns) + 2
            else:
                ws_no_conc.range((3, current_col)).value = "Sin datos"
                current_col += 3
        
        ws_no_conc.autofit()
        print("    -> Hoja 'No conciliado' exportada.")
        
        # =====================================================================
        # 3. ADICIONALES
        # =====================================================================
        ws_adic = obtener_hoja_segura(wb, 'Caja Adicion')
        if not df_adic.empty:
            ws_adic.range('A1').options(index=False).value = df_adic
            ws_adic.autofit()
        else:
            ws_adic.range('A1').value = "Sin datos de adicionales"

        # =====================================================================
        # 4. DESCUENTOS
        # =====================================================================
        ws_desc = obtener_hoja_segura(wb, 'Descuentos')
        if not df_desc.empty:
            ws_desc.range('A1').options(index=False).value = df_desc
            ws_desc.autofit()
        else:
            ws_desc.range('A1').value = "Sin datos de descuentos"

        # =====================================================================
        # 5. AUDITORÍA (NUEVO BLOQUE INTEGRADO)
        # =====================================================================
        print("    -> Exportando Auditoría...")
        ws_aud = obtener_hoja_segura(wb, 'Auditoria Estructurada')
        
        if not df_audit.empty:
            ws_aud.range('A1').options(index=False).value = df_audit
            ws_aud.autofit()
            print("    -> Auditoría exportada.")
        else:
            ws_aud.range('A1').value = "Sin datos de auditoría"

        # --- GUARDAR Y CERRAR ---
        wb.save()
        print("    -> ✅ Exportación al MACRO completada.")
        
    except Exception as e:
        print(f"    [ERROR] xlwings: {e}")
        import traceback
        traceback.print_exc()
    finally:
        try:
            wb.close()
            app.quit()
        except:
            pass
        # =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

if __name__ == "__main__":
    try:
        print("\n=== INICIANDO PROCESO ===")
        
        # Validar que se hayan seleccionado las carpetas críticas
        if not folder_salida:
            print("❌ ERROR: No seleccionaste carpeta de salida. El programa se detendrá.")
            exit()

        # 1. CARGA (AGREGAMOS df_audit AL FINAL)
        df_mp, df_nave, df_sys, df_t, df_adic, df_desc, df_audit= cargar_datos_normalizados()
        
        # --- DEFINICIÓN DINÁMICA DE RUTAS DE SALIDA ---
        path_turnos_output = os.path.join(folder_salida, "Turnos_Procesados.xlsx")
        path_informe_limpio = os.path.join(folder_salida, "Informe_Invernadas_Procesado.xlsx")
        
        # Guardado 1: Turnos
        if not df_t.empty:
            print(f"   Guardando Turnos en: {path_turnos_output}")
            df_t.to_excel(path_turnos_output, sheet_name='Turnos_Procesados', index=False)

        # Guardado 2: Informe Limpio (AGREGAMOS df_audit)
        print(f"   Guardando Informe Procesado en: {path_informe_limpio}")
        with pd.ExcelWriter(path_informe_limpio, engine='xlsxwriter', datetime_format='dd/mm/yyyy hh:mm') as writer_clean:
            if not df_mp.empty: df_mp.to_excel(writer_clean, sheet_name='MP_Normalizado', index=False)
            # CAMBIO AQUÍ: Nombre de la hoja
            if not df_nave.empty: df_nave.to_excel(writer_clean, sheet_name='Fiserv_Normalizado', index=False) 
            if not df_sys.empty: df_sys.to_excel(writer_clean, sheet_name='Sistema_Normalizado', index=False)
            if not df_adic.empty: df_adic.to_excel(writer_clean, sheet_name='Adicionales_Normalizado', index=False)
            if not df_desc.empty: df_desc.to_excel(writer_clean, sheet_name='Descuentos_Normalizado', index=False)
            # --- NUEVO ---
            if not df_audit.empty: df_audit.to_excel(writer_clean, sheet_name='Auditoria_Normalizado', index=False)
            

        # 3. Conciliación
        df_mp = recuperar_clasificaciones(df_mp, path_output_final, 'MP Conciliado')
        df_nave = recuperar_clasificaciones(df_nave, path_output_final, 'Fiserv Conciliado')
        
        # --- Conciliación MP ---
        if not df_mp.empty:
            correr_conciliacion(df_mp, df_sys, "MP vs QR (Ideal)", TARGET_QR)
            correr_conciliacion(df_mp, df_sys, "MP vs Efectivo (Rescate)", TARGET_EFECTIVO)
            correr_conciliacion(df_mp, df_sys, "MP vs CtaCte (Rescate)", TARGET_CTACTE)
        
        # --- Conciliación NAVE ---
        if not df_nave.empty:
            correr_conciliacion(df_nave, df_sys, "Fiserv vs Sistema", None)
            correr_conciliacion(df_nave, df_sys, "Fiserv vs Efectivo (Rescate)", TARGET_EFECTIVO)
            correr_conciliacion(df_nave, df_sys, "Fiserv vs CtaCte (Rescate)", TARGET_CTACTE)

       
        
        # --- Cruces Inter-Turnos ---
        if not df_mp.empty:
            correr_conciliacion_inter_turnos(df_mp, df_sys, TARGET_QR)
            
        if not df_nave.empty:
            correr_conciliacion_inter_turnos(df_nave, df_sys, None)

        # --- FASE GLOBAL ---
        print("\n=== FASE FINAL: CONCILIACIÓN CRUZADA (MP vs NAVE) ===")
        if not df_mp.empty:
            correr_conciliacion(df_mp, df_sys, "MP Global → Registros FISERV", None, ignorar_medio=True, solo_nave=True)

        if not df_nave.empty:
            correr_conciliacion_nave_invertida(df_nave, df_sys)

        # 4. PROCESAR CRUCE DE DESCUENTOS
        if not df_desc.empty:
            df_desc = procesar_cruce_descuentos(df_desc, df_sys)

        # Guardado 3: Resultado Final
        print(f"\n[GUARDANDO] Resultado Final en: {path_output_final}")
        with pd.ExcelWriter(path_output_final, engine='xlsxwriter', datetime_format='dd/mm/yyyy hh:mm') as writer:
            
            # Parche de seguridad (crear vacíos si no existen)
            if 'df_mp' not in locals() or df_mp is None or df_mp.empty:
                df_mp = pd.DataFrame(columns=['Clasificacion', 'EXTERNAL_REFERENCE', 'TOTAL_PAID_AMOUNT', 'datetime_col', 'TURNO', 'monto_col_numeric', 'Estado', 'ID_Original', 'Plataforma', 'Tipo Match', 'ID_Cruce'])
                df_mp['Estado'] = 'No Conciliado'

            if 'Plataforma' not in df_nave.columns: df_nave['Plataforma'] = 'Fiserv'
            if 'Clasificacion' not in df_nave.columns: df_nave['Clasificacion'] = ''
            if 'Tipo Match' not in df_nave.columns: df_nave['Tipo Match'] = ''

            # Generar reportes excel
            generar_reporte_plano(writer, df_mp, df_nave, df_sys)
            generar_auditoria(writer, df_mp, df_nave, df_sys)
            
            if not df_mp.empty: df_mp.to_excel(writer, sheet_name='MP Conciliado', index=False)
            if not df_nave.empty: df_nave.to_excel(writer, sheet_name='Fiserv Conciliado', index=False)
            
            if not df_sys.empty:
                df_sys_limpio = df_sys[df_sys['TURNO'] != 'Fecha Nula']
                df_sys_limpio.to_excel(writer, sheet_name='Sistema Conciliado', index=False)
            
            

            if not df_adic.empty:
                df_adic.to_excel(writer, sheet_name='Adicionales Procesados', index=False)


            if not df_desc.empty:
                df_desc.to_excel(writer, sheet_name='Descuentos Procesados', index=False)
            
            # --- NUEVO: AUDITORÍA EN EL EXCEL FINAL ---
            if not df_audit.empty:
                df_audit.to_excel(writer, sheet_name='Auditoria_Estructurada', index=False)

            # Formato columna A
            for sheet in writer.sheets.values(): sheet.set_column('A:A', 20) 
            
            # 5. Exportar al MACRO (PASAMOS df_audit)
            exportar_al_macro(df_mp, df_nave, df_sys, df_adic, df_desc, df_audit)
            
        print("\n¡Proceso Finalizado con Éxito!")

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        
    input("Presiona ENTER para salir...")


=== SELECCIÓN DE ARCHIVOS ===
Esperando selección: Selecciona el archivo de TURNOS...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/Turnos Invernadas Paula.xlsx
Esperando selección: Selecciona el archivo de MERCADO PAGO...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/collection-20260209184940-c4c0.xlsx
Esperando selección: Selecciona el archivo de FISERV...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/Transacciones_202602101047221.xlsx
Esperando selección: Selecciona el archivo de VENTAS (Sistema)...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/VENTAS 9-2 (1).XLSX
Esperando selección: Selecciona el archivo de ADICIÓN...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/ADICION 9-02 (1).XLSX
Esperando selección: Selecciona el archivo de DESCUENTOS...
✅ Seleccionado: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/DESCUENT